In [3]:
import pandas as pd, numpy as np, seaborn as sns, warnings as warn, matplotlib.pyplot as plt
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, precision_score, recall_score
from sklearn.model_selection import train_test_split , RandomizedSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import BernoulliNB
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

In [5]:
df = pd.read_csv('Churn_Modelling.csv')

In [6]:
df.head()
df.shape
df.isna().sum()

RowNumber          0
CustomerId         0
Surname            0
CreditScore        0
Geography          0
Gender             0
Age                0
Tenure             0
Balance            0
NumOfProducts      0
HasCrCard          0
IsActiveMember     0
EstimatedSalary    0
Exited             0
dtype: int64

In [7]:
# df = sns.load_dataset('tips')
df = pd.read_csv('Churn_Modelling.csv')

In [8]:
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [9]:
# X = df.drop('time', axis = 1)
# y = df['time']
X = df.drop(['Exited','CustomerId', 'Surname', 'RowNumber'], axis = 1)
y = df['Exited']

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=.25, random_state = 42)

In [16]:
# label_encoder = LabelEncoder()
# y_train = label_encoder.fit_transform(y_train)
# y_test = label_encoder.transform(y_test)

In [11]:
# cat_features = [col for col in X.columns if df[col].dtype == 'category']
# num_features = [col for col in X.columns if df[col].dtype != 'category']
cat_features = [col for col in X.columns if df[col].dtype == 'O']
num_features = [col for col in X.columns if df[col].dtype != 'O']

In [12]:
num_features

['CreditScore',
 'Age',
 'Tenure',
 'Balance',
 'NumOfProducts',
 'HasCrCard',
 'IsActiveMember',
 'EstimatedSalary']

In [13]:
cat_features

['Geography', 'Gender']

In [14]:
num_pipeline = Pipeline(steps = [('imputer', SimpleImputer(strategy = 'median')), ('scaler',StandardScaler())])
cat_pipeline = Pipeline(steps = [('imputer', SimpleImputer(strategy = 'most_frequent')), ('encoder', OneHotEncoder())])

In [15]:
preprocessor = ColumnTransformer([('num_pipeline', num_pipeline, num_features), ('cat_pipeline', cat_pipeline, cat_features)])

In [16]:
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

In [17]:
X = preprocessor.fit_transform(X)
X_train,X_test,y_train, y_test = train_test_split(X, y, test_size = 0.25, random_state = 42)

In [18]:
models = {'svc' : SVC(), 'dtc': DecisionTreeClassifier(), 'rfc' : RandomForestClassifier(), 'bnb': BernoulliNB()}

In [19]:
for model_name, model in models.items():
    model.fit(X_train,y_train)
    y_pred = model.predict(X_test)
    print(model_name)
    print(accuracy_score(y_test,y_pred))
    print(confusion_matrix(y_test,y_pred))
    print(recall_score(y_test,y_pred))
    print(precision_score(y_test,y_pred))
    print(classification_report(y_test,y_pred))
    print('*'*50)

svc
0.8572
[[1954   49]
 [ 308  189]]
0.38028169014084506
0.7941176470588235
              precision    recall  f1-score   support

           0       0.86      0.98      0.92      2003
           1       0.79      0.38      0.51       497

    accuracy                           0.86      2500
   macro avg       0.83      0.68      0.72      2500
weighted avg       0.85      0.86      0.84      2500

**************************************************
dtc
0.7972
[[1741  262]
 [ 245  252]]
0.5070422535211268
0.490272373540856
              precision    recall  f1-score   support

           0       0.88      0.87      0.87      2003
           1       0.49      0.51      0.50       497

    accuracy                           0.80      2500
   macro avg       0.68      0.69      0.69      2500
weighted avg       0.80      0.80      0.80      2500

**************************************************
rfc
0.868
[[1935   68]
 [ 262  235]]
0.47283702213279677
0.7755775577557755
              pr

In [33]:
params={'max_depth':[3,5,10,None],
              'n_estimators':[100,200,300],
               'criterion':['gini','entropy']
              }
rfc = RandomForestClassifier()
random_cv = RandomizedSearchCV(rfc, param_distributions= params, cv= 5 , verbose = 3, scoring = 'accuracy')
random_cv.fit(X_train,y_train)
print(random_cv.best_score_)
print(random_cv.best_params_)

Fitting 5 folds for each of 10 candidates, totalling 50 fits
[CV 1/5] END criterion=gini, max_depth=5, n_estimators=300;, score=0.795 total time=   1.1s
[CV 2/5] END criterion=gini, max_depth=5, n_estimators=300;, score=0.795 total time=   1.4s
[CV 3/5] END criterion=gini, max_depth=5, n_estimators=300;, score=0.795 total time=   1.0s
[CV 4/5] END criterion=gini, max_depth=5, n_estimators=300;, score=0.795 total time=   1.3s
[CV 5/5] END criterion=gini, max_depth=5, n_estimators=300;, score=0.795 total time=   1.2s
[CV 1/5] END criterion=gini, max_depth=None, n_estimators=300;, score=0.851 total time=  14.5s
[CV 2/5] END criterion=gini, max_depth=None, n_estimators=300;, score=0.841 total time=  14.8s
[CV 3/5] END criterion=gini, max_depth=None, n_estimators=300;, score=0.858 total time=  18.3s
[CV 4/5] END criterion=gini, max_depth=None, n_estimators=300;, score=0.850 total time=  19.6s
[CV 5/5] END criterion=gini, max_depth=None, n_estimators=300;, score=0.845 total time=  18.8s
[CV 

In [35]:
rfc = RandomForestClassifier(n_estimators= 300, criterion= 'gini')
rfc.fit(X_train,y_train)
y_pred = rfc.predict(X_test)
print(accuracy_score(y_test,y_pred))
print(confusion_matrix(y_test,y_pred))
print(recall_score(y_test,y_pred))
print(precision_score(y_test,y_pred))
print(classification_report(y_test,y_pred))

0.8576
[[1974   29]
 [ 327  170]]
0.3420523138832998
0.8542713567839196
              precision    recall  f1-score   support

           0       0.86      0.99      0.92      2003
           1       0.85      0.34      0.49       497

    accuracy                           0.86      2500
   macro avg       0.86      0.66      0.70      2500
weighted avg       0.86      0.86      0.83      2500



## RandomForest Classifier Implementation With Pipelines And Hyperparameter Tuning

In [76]:
import seaborn as sns
df=sns.load_dataset('tips')
df.head()

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


In [77]:
df['time'].unique()

['Dinner', 'Lunch']
Categories (2, object): ['Lunch', 'Dinner']

In [78]:
## Handling Missing Values
## handling Categorical features
## handling outliers
## Feature scaling
## Automating the entire

In [3]:
df.head()

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 244 entries, 0 to 243
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   total_bill  244 non-null    float64 
 1   tip         244 non-null    float64 
 2   sex         244 non-null    category
 3   smoker      244 non-null    category
 4   day         244 non-null    category
 5   time        244 non-null    category
 6   size        244 non-null    int64   
dtypes: category(4), float64(2), int64(1)
memory usage: 7.4 KB


In [6]:
from sklearn.preprocessing import LabelEncoder
encoder=LabelEncoder()
df['time']=encoder.fit_transform(df['time'])

In [8]:
df['time'].value_counts()

0    176
1     68
Name: time, dtype: int64

In [9]:
## independent and dependent features
X=df.drop(labels=['time'],axis=1)
y=df['time']

In [10]:
X.head()

,total_bill,tip,sex,smoker,day,size
0,16.99,1.01,Female,No,Sun,2
1,10.34,1.66,Male,No,Sun,3
2,21.01,3.50,Male,No,Sun,3
3,23.68,3.31,Male,No,Sun,2
4,24.59,3.61,Female,No,Sun,4


In [11]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.20,random_state=42)

In [12]:
X_train.head()

,total_bill,tip,sex,smoker,day,size
228,13.28,2.72,Male,No,Sat,2
208,24.27,2.03,Male,Yes,Sat,2
96,27.28,4.00,Male,Yes,Fri,2
167,31.71,4.50,Male,No,Sun,4
84,15.98,2.03,Male,No,Thur,2


In [13]:
from sklearn.impute import SimpleImputer ## Handling Missing Values
from sklearn.preprocessing import OneHotEncoder## handling Categorical features
from sklearn.preprocessing import StandardScaler## Feature scaling
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
## Automating the entire

In [14]:
categorical_cols = ['sex', 'smoker','day']
numerical_cols = ['total_bill', 'tip','size']

In [15]:
## Feature Engineering Automation
num_pipeline=Pipeline(
    steps=[
        ('imputer',SimpleImputer(strategy='median')), ##missing values
        ('scaler',StandardScaler())## feature scaling 
    ]

)

#categorical Pipeline
cat_pipeline=Pipeline(
                steps=[
                ('imputer',SimpleImputer(strategy='most_frequent')), ## handling Missing values
                ('onehotencoder',OneHotEncoder()) ## Categorical features to numerical
                ]

            )  


In [16]:
preprocessor=ColumnTransformer([
    ('num_pipeline',num_pipeline,numerical_cols),
    ('cat_pipeline',cat_pipeline,categorical_cols)

])

In [18]:
X_train=preprocessor.fit_transform(X_train)
X_test=preprocessor.transform(X_test)

In [29]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC

In [30]:
## Automate Model Training Process
models={
    'Random Forest':RandomForestClassifier(),
    'Decision Tree':DecisionTreeClassifier(),
    'SVC':SVC()

}

In [21]:
from sklearn.metrics import accuracy_score

In [22]:
def evaluate_model(X_train,y_train,X_test,y_test,models):
    
    report = {}
    for i in range(len(models)):
        model = list(models.values())[i]
        # Train model
        model.fit(X_train,y_train)

            

        # Predict Testing data
        y_test_pred =model.predict(X_test)

        # Get accuracy for test data prediction
       
        test_model_score = accuracy_score(y_test,y_test_pred)

        report[list(models.keys())[i]] =  test_model_score
            

            
    return report

In [31]:
evaluate_model(X_train,y_train,X_test,y_test,models)

{'Random Forest': 0.9591836734693877,
 'Decision Tree': 0.9387755102040817,
 'SVC': 0.9591836734693877}

In [32]:
classifier=RandomForestClassifier()

In [34]:
## Hypeparameter Tuning
params={'max_depth':[3,5,10,None],
              'n_estimators':[100,200,300],
               'criterion':['gini','entropy']
              }

In [33]:
from sklearn.model_selection import RandomizedSearchCV

In [36]:
cv=RandomizedSearchCV(classifier,param_distributions=params,scoring='accuracy',cv=5,verbose=3)
cv.fit(X_train,y_train)

Fitting 5 folds for each of 10 candidates, totalling 50 fits
[CV 1/5] END criterion=entropy, max_depth=None, n_estimators=100;, score=0.974 total time=   0.2s
[CV 2/5] END criterion=entropy, max_depth=None, n_estimators=100;, score=0.923 total time=   0.2s
[CV 3/5] END criterion=entropy, max_depth=None, n_estimators=100;, score=1.000 total time=   0.2s
[CV 4/5] END criterion=entropy, max_depth=None, n_estimators=100;, score=0.923 total time=   0.2s
[CV 5/5] END criterion=entropy, max_depth=None, n_estimators=100;, score=0.923 total time=   0.2s
[CV 1/5] END criterion=gini, max_depth=5, n_estimators=100;, score=0.974 total time=   0.2s
[CV 2/5] END criterion=gini, max_depth=5, n_estimators=100;, score=0.923 total time=   0.2s
[CV 3/5] END criterion=gini, max_depth=5, n_estimators=100;, score=0.974 total time=   0.2s
[CV 4/5] END criterion=gini, max_depth=5, n_estimators=100;, score=0.949 total time=   0.2s
[CV 5/5] END criterion=gini, max_depth=5, n_estimators=100;, score=0.923 total ti

RandomizedSearchCV(cv=5, estimator=RandomForestClassifier(),
                   param_distributions={'criterion': ['gini', 'entropy'],
                                        'max_depth': [3, 5, 10, None],
                                        'n_estimators': [100, 200, 300]},
                   scoring='accuracy', verbose=3)

In [37]:
cv.best_params_

{'n_estimators': 100, 'max_depth': 10, 'criterion': 'gini'}

In [ ]:
## Internal Assignment
## Decision Tree Regression